# MedGemma — Image processing (Colab, CPU runtime)

Processes DMID `.tif` images to `.png` with the Mammo-CLIP pipeline (5-px border
crop, intensity rescale, `ExtractBreast`, resize 912x1520). VinDr images are
downloaded as Mammo-CLIP PNGs and need no processing here.

Output: `data/dmid/images_png/{stem}.png` on Drive.

In [ ]:
import glob
import os
from pathlib import Path

import cv2
import numpy as np
from tqdm.auto import tqdm

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
ROOT_DIR = Path("/content/drive/MyDrive/MedGemma2026/main")
DATA_DIR = ROOT_DIR / "data"

DMID_TIF_DIR = DATA_DIR / "dmid" / "images-original"
DMID_PNG_DIR = DATA_DIR / "dmid" / "images-processed"
SIZE = (912, 1520)

In [ ]:
def np_CountUpContinuingOnes(b_arr):
    left = np.arange(len(b_arr))
    left[b_arr > 0] = 0
    left = np.maximum.accumulate(left)

    rev_arr = b_arr[::-1]
    right = np.arange(len(rev_arr))
    right[rev_arr > 0] = 0
    right = np.maximum.accumulate(right)
    right = len(rev_arr) - 1 - right[::-1]

    return right - left - 1


def ExtractBreast(img):
    img_copy = img.copy()
    img = np.where(img <= 40, 0, img)  # To detect backgrounds easily
    height, _ = img.shape

    y_a = height // 2 + int(height * 0.4)
    y_b = height // 2 - int(height * 0.4)
    b_arr = img[y_b:y_a].std(axis=0) != 0
    continuing_ones = np_CountUpContinuingOnes(b_arr)
    col_ind = np.where(continuing_ones == continuing_ones.max())[0]
    img = img[:, col_ind]

    _, width = img.shape
    x_a = width // 2 + int(width * 0.4)
    x_b = width // 2 - int(width * 0.4)
    b_arr = img[:, x_b:x_a].std(axis=1) != 0
    continuing_ones = np_CountUpContinuingOnes(b_arr)
    row_ind = np.where(continuing_ones == continuing_ones.max())[0]

    return img_copy[row_ind][:, col_ind]


def process_image(in_path, out_path, SIZE=(912, 1520)):
    data = cv2.imread(str(in_path), cv2.IMREAD_GRAYSCALE)
    if data is None:
        raise FileNotFoundError(in_path)

    data = data[5:-5, 5:-5]  # border crop
    data = data - np.min(data)  # rescale intensity to full 8-bit range
    data = data / np.max(data)
    data = (data * 255).astype(np.uint8)

    img = ExtractBreast(data)  # crop to the breast region
    img = cv2.resize(img, SIZE, interpolation=cv2.INTER_AREA)  # resize
    cv2.imwrite(str(out_path), img)

In [6]:
# Idempotent: skips outputs that already exist.
DMID_PNG_DIR.mkdir(parents=True, exist_ok=True)
tif_paths = sorted(
    glob.glob(os.path.join(str(DMID_TIF_DIR), "*.tif"))
    + glob.glob(os.path.join(str(DMID_TIF_DIR), "*.tiff"))
)
print(f"DMID .tif images to process: {len(tif_paths)}")

processed = skipped = 0
for in_path in tqdm(tif_paths, desc="DMID .tif -> .png"):
    out_path = DMID_PNG_DIR / (Path(in_path).stem + ".png")
    if out_path.exists():
        skipped += 1
        continue
    process_image(in_path, out_path, SIZE=SIZE)
    processed += 1

print(f"processed={processed}  skipped(existing)={skipped}  -> {DMID_PNG_DIR}")

DMID .tif images to process: 511


DMID .tif -> .png:   0%|          | 0/511 [00:00<?, ?it/s]

processed=413  skipped(existing)=98  -> /content/drive/MyDrive/MedGemma2026/main/data/dmid/images-processed
